In [1]:
!pip install -q graph-part
!apt-get install -y emboss -q   # provides `needleall`, GraphPart's default alignment backend

# Confirm the CLI's actual expected arguments directly from --help, rather than
# trusting documentation from memory -- same standard used for CD-HIT in this project.
!graphpart needle --help

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 586.5 kB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  dialign emboss-data emboss-lib libhpdf-2.3.0 primer3
Suggested packages:
  emboss-doc emboss-test embassy clustalw ncbi-epcr
The following NEW packages will be installed:
  dialign emboss emboss-data emboss-lib libhpdf-2.3.0 primer3
0 upgraded, 6 newly installed, 0 to remove and 24 not upgraded.
Need to get 65.7 MB of archives.
After this operation, 500 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 dialign amd64 2.2.1-11 [139 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhpdf-2.3.0 amd64 2.3.0+dfsg-1build1 [335 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 emboss-lib amd64 6.6.0+dfsg-11ubuntu1 [2,971 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 emboss-data all 

In [3]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/AOP-ProCon"
!mkdir -p "{BASE}/data/raw" "{BASE}/data/processed" "{BASE}/data/embeddings" "{BASE}/results"

import pandas as pd
import numpy as np
import requests, re, subprocess

pd.set_option("display.max_colwidth", 80)

/bin/bash: line 1: nvidia-smi: command not found
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd, subprocess

aop_benchpos = pd.read_csv(f"{BASE}/data/processed/AOP-BenchPos.csv")

def run_graphpart(df, out_prefix, threshold=0.40, partitions=2, threads=2):
    fasta_path = f"/content/{out_prefix}.fasta"
    out_csv = f"/content/{out_prefix}_assignments.csv"
    with open(fasta_path, "w") as f:
        for i, row in df.reset_index(drop=True).iterrows():
            f.write(f">seq{i}|label={row['tier']}\n{row['sequence']}\n")

    result = subprocess.run([
        "graphpart", "needle",
        "--fasta-file", fasta_path,
        "--threshold", str(threshold),
        "--out-file", out_csv,
        "--labels-name", "label",
        "--partitions", str(partitions),
        "--threads", str(threads),
    ], capture_output=True, text=True)

    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)
    print("Return code:", result.returncode)
    return out_csv

# Small test first: Tier 2 only (150 sequences, ~11k pairs) -- fast enough to sanity-check
# the command actually works before running it on the full 1,639-sequence pool.
tier2_only = aop_benchpos[aop_benchpos["tier"] == "Tier2"]
test_out = run_graphpart(tier2_only, "graphpart_test_tier2")

STDOUT:
        val  num  lim
Tier2    0  150   75
Computing pairwise sequence identities.
Pairwise alignment executed in 2.06 seconds.
Full graph nr. of edges: 309
75 [75]
Initialization mode slow-nn
Edge iteration completed in 0.00 seconds.
Edge sorting competed at 0.01 seconds.
1
cluster     0.0   1.0  label  mean  count
label-val                                
0          79.0  71.0  Tier2  75.0  150.0

Partitioning score: 74.89325737341113

cluster     0.0   1.0  label  mean  count
label-val                                
0          79.0  71.0  Tier2  75.0  150.0

Partitioning score: 74.89325737341113

Currently have this many samples: 150
After removal we have this many samples: 150
cluster     0.0   1.0  label  mean  count
label-val                                
0          79.0  71.0  Tier2  75.0  150.0

Partitioning score: 74.89325737341113

Graph-Part executed in 2.19 seconds.

STDERR:
 
 83%|████████▎ | 18756/22500 [00:02<00:00, 9314.51it/s]

Initializing: 0it [00:00, ?it/

In [5]:
full_out = run_graphpart(aop_benchpos, "graphpart_full", partitions=2, threads=4)
graphpart_result = pd.read_csv(full_out)
print(graphpart_result.shape)
graphpart_result.head()

STDOUT:
        val  num  lim
Tier1    0  631  315
Tier2    1  150   75
Tier3    2  858  429
Computing pairwise sequence identities.
Pairwise alignment executed in 93.34 seconds.
Full graph nr. of edges: 14985
819 [315  75 429]
Initialization mode slow-nn
Edge iteration completed in 0.01 seconds.
Edge sorting competed at 0.01 seconds.
3
cluster      0.0    1.0  label   mean  count
label-val                                   
0          316.0  315.0  Tier1  315.5  631.0
1           75.0   75.0  Tier2   75.0  150.0
2          429.0  429.0  Tier3  429.0  858.0

Partitioning score: 216.5238752291125

cluster      0.0    1.0  label   mean  count
label-val                                   
0          316.0  315.0  Tier1  315.5  631.0
1           75.0   75.0  Tier2   75.0  150.0
2          429.0  429.0  Tier3  429.0  858.0

Partitioning score: 216.5238752291125

Currently have this many samples: 1639
!  seq1 seq2 {'metric': 0.571}  !
Need to remove! Currently have this many samples: 1639
Min

,AC,priority,label-val,between_connectivity,cluster
0,seq0,False,0,0,0.0
1,seq1,False,0,0,0.0
2,seq2,False,0,0,0.0
3,seq3,False,0,0,1.0
4,seq4,False,0,0,1.0


In [6]:
import pandas as pd

aop_benchpos = pd.read_csv(f"{BASE}/data/processed/AOP-BenchPos.csv").reset_index(drop=True)
graphpart_result = pd.read_csv("/content/graphpart_full_assignments.csv")

# Map GraphPart's "seqN" IDs back to the original row index
kept_indices = graphpart_result["AC"].str.replace("seq", "", regex=False).astype(int)
kept_mask = aop_benchpos.index.isin(kept_indices)

dropped = aop_benchpos[~kept_mask].copy()
kept = aop_benchpos[kept_mask].copy()
dropped["seq_len"] = dropped["sequence"].str.len()
kept["seq_len"] = kept["sequence"].str.len()

print("Dropped:", len(dropped), "| Kept:", len(kept))

print("\nDropped sequence length distribution:")
print(dropped["seq_len"].describe())
print("\nKept sequence length distribution:")
print(kept["seq_len"].describe())

print("\nDropped length value counts (sorted):")
print(dropped["seq_len"].value_counts().sort_index())

# Same comparison used to catch the CD-HIT length-3 bug: is <10 AA over-represented among the dropped?
short_dropped = (dropped["seq_len"] < 10).mean()
short_kept = (kept["seq_len"] < 10).mean()
print(f"\nFraction <10 AA -- Dropped: {short_dropped:.1%} | Kept: {short_kept:.1%}")

Dropped: 151 | Kept: 1488

Dropped sequence length distribution:
count    151.000000
mean       6.013245
std        1.608257
min        3.000000
25%        5.000000
50%        6.000000
75%        7.000000
max       14.000000
Name: seq_len, dtype: float64

Kept sequence length distribution:
count    1488.000000
mean        7.418011
std         6.131160
min         3.000000
25%         3.000000
50%         5.000000
75%         9.000000
max        49.000000
Name: seq_len, dtype: float64

Dropped length value counts (sorted):
seq_len
3      7
4     17
5     29
6     47
7     34
8      6
9      9
11     1
14     1
Name: count, dtype: int64

Fraction <10 AA -- Dropped: 98.7% | Kept: 77.9%


In [7]:
full_out2 = run_graphpart(aop_benchpos, "graphpart_full_run2", partitions=2, threads=4)
graphpart_result2 = pd.read_csv(full_out2)
print(graphpart_result2.shape)

STDOUT:
        val  num  lim
Tier1    0  631  315
Tier2    1  150   75
Tier3    2  858  429
Computing pairwise sequence identities.
Pairwise alignment executed in 90.29 seconds.
Full graph nr. of edges: 14985
819 [315  75 429]
Initialization mode slow-nn
Edge iteration completed in 0.02 seconds.
Edge sorting competed at 0.02 seconds.
3
cluster      0.0    1.0  label   mean  count
label-val                                   
0          316.0  315.0  Tier1  315.5  631.0
1           75.0   75.0  Tier2   75.0  150.0
2          429.0  429.0  Tier3  429.0  858.0

Partitioning score: 216.5238752291125

cluster      0.0    1.0  label   mean  count
label-val                                   
0          316.0  315.0  Tier1  315.5  631.0
1           75.0   75.0  Tier2   75.0  150.0
2          429.0  429.0  Tier3  429.0  858.0

Partitioning score: 216.5238752291125

Currently have this many samples: 1639
!  seq1 seq2 {'metric': 0.571}  !
Need to remove! Currently have this many samples: 1639
Min

In [8]:
r1 = pd.read_csv("/content/graphpart_full_assignments.csv")
r2 = pd.read_csv("/content/graphpart_full_run2_assignments.csv")

print("Run1 shape:", r1.shape, "| Run2 shape:", r2.shape)

set1 = set(r1["AC"])
set2 = set(r2["AC"])
print("Same set of kept/dropped sequences:", set1 == set2)
print("Only in run1 (dropped in run2):", len(set1 - set2))
print("Only in run2 (dropped in run1):", len(set2 - set1))

merged = r1[["AC", "cluster"]].merge(r2[["AC", "cluster"]], on="AC", suffixes=("_run1", "_run2"), how="outer")
merged["cluster_run2_swapped"] = 1 - merged["cluster_run2"]

match_asis = (merged["cluster_run1"] == merged["cluster_run2"]).mean()
match_swapped = (merged["cluster_run1"] == merged["cluster_run2_swapped"]).mean()
print(f"\nAgreement (as-is labels): {match_asis:.1%}")
print(f"Agreement (swapped labels, 0<->1): {match_swapped:.1%}")
print(f"Best-case agreement: {max(match_asis, match_swapped):.1%}")

mismatches = merged[(merged['cluster_run1'] != merged['cluster_run2']) & (merged['cluster_run1'] != merged['cluster_run2_swapped'])]
print("True mismatches (agree with neither labeling):", len(mismatches))

Run1 shape: (1488, 5) | Run2 shape: (1488, 5)
Same set of kept/dropped sequences: True
Only in run1 (dropped in run2): 0
Only in run2 (dropped in run1): 0

Agreement (as-is labels): 100.0%
Agreement (swapped labels, 0<->1): 0.0%
Best-case agreement: 100.0%
True mismatches (agree with neither labeling): 0
